## Developing with SQS and SNS

### Introduction: Messaging with SQS and SNS

Welcome back! So far, you have learned how to work with AWS services like S3 and DynamoDB using Python and the boto3 library. In this lesson, we will explore two more important AWS services: SQS (Simple Queue Service) and SNS (Simple Notification Service).

In modern cloud applications, different parts of your system often need to communicate with each other. Sometimes, you want to send a message from one part of your application to another, but you do not want them to be tightly connected. This is where messaging services like SQS and SNS come in. They help you build applications that are more flexible, reliable, and easier to scale.

By the end of this lesson, you will know how to:

* Create an SQS queue and an SNS topic using Python and boto3.
* Send messages to an SQS queue (acting as a producer).
* Receive and process messages from the queue (acting as a consumer).
* Publish results to an SNS topic.

Let's get started!

---

## Recall: Using boto3 to Work with AWS Services

Before we dive in, let's quickly remind ourselves about boto3. In previous lessons, you used boto3 to interact with AWS services like S3 and DynamoDB. boto3 is the official AWS SDK for Python, and it allows you to create, read, update, and delete AWS resources using Python code.

On CodeSignal, boto3 is already installed, so you do not need to worry about setting it up. You also learned that AWS credentials are needed to access AWS services. On CodeSignal, these are handled for you, but in real-world projects, you would use IAM roles or environment variables to provide credentials securely.

Now, let's see how to use boto3 to work with SQS and SNS.

---

## Creating SQS Queues and SNS Topics

The first step is to create the messaging resources: an SQS queue and an SNS topic. These are the places where messages will be sent and received.

Let's start by importing the necessary libraries and creating boto3 clients for SQS and SNS:

```python
import boto3
import uuid

sqs = boto3.client('sqs')
sns = boto3.client('sns')
```

* `boto3.client('sqs')` creates a client to interact with SQS.
* `boto3.client('sns')` creates a client to interact with SNS.
* We also import `uuid` to help us create unique names for our resources.

Now, let's create a new SQS queue and an SNS topic. We will use unique names to avoid conflicts:

```python
q_name = f"orders-{uuid.uuid4().hex[:8]}"
t_name = f"order-events-{uuid.uuid4().hex[:8]}"
q_url = sqs.create_queue(QueueName=q_name)["QueueUrl"]
topic_arn = sns.create_topic(Name=t_name)["TopicArn"]

print("Created Queue URL:", q_url)
print("Created Topic ARN:", topic_arn)
```

* `sqs.create_queue(QueueName=q_name)` creates a new SQS queue and returns its details. We extract the `QueueUrl` from the response.
* `sns.create_topic(Name=t_name)` creates a new SNS topic and returns its details. We extract the `TopicArn` from the response.

**Example Output:**

```text
Created Queue URL: https://sqs.us-east-1.amazonaws.com/123456789012/orders-1a2b3c4d
Created Topic ARN: arn:aws:sns:us-east-1:123456789012:order-events-5e6f7g8h
```

Now you have an SQS queue and an SNS topic ready to use!

---

## Sending Messages to SQS

Next, let's act as a producer and send messages to the SQS queue. In this example, we will send two simple order messages.

First, let's prepare the messages and send them to the queue:

```python
import json

orders = [
    {"order_id": "o-001", "amount": 42.0},
    {"order_id": "o-002", "amount": 99.5}
]

for o in orders:
    sqs.send_message(QueueUrl=q_url, MessageBody=json.dumps(o))
    print("Sent:", o["order_id"])
```

* We create a list of orders, each represented as a dictionary.
* We use `json.dumps(o)` to convert each order to a JSON string, which is the format SQS expects.
* `sqs.send_message(QueueUrl=q_url, MessageBody=...)` sends the message to the queue.

**Example Output:**

```text
Sent: o-001
Sent: o-002
```

This is how you send messages to an SQS queue. The producer part is now complete.

---

## Consuming and Processing Messages from SQS

Now, let's act as a consumer. We will receive messages from the SQS queue, process them, and then delete them from the queue.

Here's how you can receive and process messages:

```python
import time

def process_order(body):
    return {"order_id": body["order_id"], "status": "processed", "amount": body["amount"]}

deadline = time.time() + 20  # Run for up to 20 seconds
while time.time() < deadline:
    resp = sqs.receive_message(QueueUrl=q_url, MaxNumberOfMessages=5, WaitTimeSeconds=5)
    for m in resp.get("Messages", []):
        body = json.loads(m["Body"])
        result = process_order(body)
        print("Processed:", result["order_id"])
        sqs.delete_message(QueueUrl=q_url, ReceiptHandle=m["ReceiptHandle"])
```

* `sqs.receive_message(...)` fetches up to 5 messages from the queue, waiting up to 5 seconds if needed.
* We use `json.loads(m["Body"])` to convert the message back to a Python dictionary.
* The `process_order` function simulates processing the order.
* After processing, we call `sqs.delete_message(...)` to remove the message from the queue. This is important to prevent the same message from being processed again.

**Example Output:**

```text
Processed: o-001
Processed: o-002
```

This completes the consumer part. You have now received and processed messages from SQS.

---

## Publishing Results to SNS

After processing each order, let's publish the result to the SNS topic. This is useful if you want to notify other parts of your system or trigger further actions.

Here's how you can publish a message to SNS:

```python
for m in resp.get("Messages", []):
    body = json.loads(m["Body"])
    result = process_order(body)
    sns.publish(TopicArn=topic_arn, Message=json.dumps(result))
    sqs.delete_message(QueueUrl=q_url, ReceiptHandle=m["ReceiptHandle"])
    print("Processed and published:", result["order_id"])
```

* `sns.publish(TopicArn=topic_arn, Message=json.dumps(result))` sends the processed order to the SNS topic.
* This allows other systems or users subscribed to the topic to receive notifications.

**Example Output:**

```text
Processed and published: o-001
Processed and published: o-002
```

Now, every processed order is published to SNS, making your application more flexible and event-driven.

---

## Summary and Next Steps

In this lesson, you learned how to:

* Create an SQS queue and an SNS topic using boto3.
* Send messages to the SQS queue as a producer.
* Receive, process, and delete messages from the queue as a consumer.
* Publish processed results to an SNS topic.

This producer/consumer pattern is a powerful way to build decoupled and scalable applications in AWS. You are now ready to practice these skills with hands-on exercises. Try creating your own queues and topics, sending and processing messages, and publishing results to SNS. This will help you become more comfortable with event-driven development in AWS. Good luck!

## Creating Your First AWS Resources

Now that you've learned about SQS and SNS, it's time to practice creating these messaging resources yourself. You'll work with the fundamental skill of extracting important information from AWS API responses.

Your task is to complete the code that creates an SQS queue and an SNS topic. The starter code already makes the API calls to create these resources, but you need to extract the correct values from the responses.

You need to:

* Extract the `QueueUrl` from the SQS `create_queue()` response.
* Extract the `TopicArn` from the SNS `create_topic()` response.

AWS API responses come back as dictionaries, and you need to use the right keys to get the values you want. Look at the TODO comments in the code — they will guide you to the exact spots where you need to add the missing lines.

Once you complete this exercise, you'll have the foundation needed to build messaging applications with AWS services.

```python
import boto3
import uuid

# Initialize AWS clients
sqs = boto3.client('sqs')
sns = boto3.client('sns')

# Generate unique names for resources
q_name = f"orders-{uuid.uuid4().hex[:8]}"
t_name = f"order-events-{uuid.uuid4().hex[:8]}"

# Create SQS queue and SNS topic
queue_response = sqs.create_queue(QueueName=q_name)
topic_response = sns.create_topic(Name=t_name)

# Extract the important values from responses
# TODO: Extract the QueueUrl from the queue_response
q_url = None
# TODO: Extract the TopicArn from the topic_response
topic_arn = None

# Print the created resources
print("Created Queue URL:", q_url)
print("Created Topic ARN:", topic_arn)
```

Here is the completed code with the `QueueUrl` and `TopicArn` extracted from their respective responses:

```python
import boto3
import uuid

# Initialize AWS clients
sqs = boto3.client('sqs')
sns = boto3.client('sns')

# Generate unique names for resources
q_name = f"orders-{uuid.uuid4().hex[:8]}"
t_name = f"order-events-{uuid.uuid4().hex[:8]}"

# Create SQS queue and SNS topic
queue_response = sqs.create_queue(QueueName=q_name)
topic_response = sns.create_topic(Name=t_name)

# Extract the important values from responses
q_url = queue_response["QueueUrl"]
topic_arn = topic_response["TopicArn"]

# Print the created resources
print("Created Queue URL:", q_url)
print("Created Topic ARN:", topic_arn)
```

## Fix the Message Producer Bug

Nice work on creating your first SQS queue and SNS topic! Now it's time to put those resources to use by sending messages as a producer.

In this exercise, you'll work with code that attempts to send order messages to an SQS queue, but there's a bug preventing the messages from being sent correctly. The code creates a queue, prepares a list of orders, and tries to send them to SQS, but something is wrong with the message format.

Your job is to find and fix the bug in the producer code. Remember that SQS has specific requirements for how message bodies should be formatted when you call `send_message()`.

The fix should be small — only one line needs to be changed. Once you solve this, you'll have successfully implemented the producer side of the messaging pattern and will be ready to tackle more complex scenarios.

```python
import boto3
import uuid
import json

# Initialize SQS client
sqs = boto3.client('sqs')

# Create a unique queue name and create the queue
q_name = f"orders-{uuid.uuid4().hex[:8]}"
q_url = sqs.create_queue(QueueName=q_name)["QueueUrl"]
print("Created Queue URL:", q_url)

# Prepare order messages
orders = [
    {"order_id": "o-001", "amount": 42.0},
    {"order_id": "o-002", "amount": 99.5},
    {"order_id": "o-003", "amount": 15.25}
]

print("\n--- Sending Orders ---")

# Send each order to the SQS queue
for order in orders:
    sqs.send_message(QueueUrl=q_url, MessageBody=order)
    print("Sent:", order["order_id"])

print("\n--- All orders sent successfully ---")
```

Here is the completed code with `MessageBody` fixed to send a JSON string instead of a raw dictionary:

```python
import boto3
import uuid
import json

# Initialize SQS client
sqs = boto3.client('sqs')

# Create a unique queue name and create the queue
q_name = f"orders-{uuid.uuid4().hex[:8]}"
q_url = sqs.create_queue(QueueName=q_name)["QueueUrl"]
print("Created Queue URL:", q_url)

# Prepare order messages
orders = [
    {"order_id": "o-001", "amount": 42.0},
    {"order_id": "o-002", "amount": 99.5},
    {"order_id": "o-003", "amount": 15.25}
]

print("\n--- Sending Orders ---")

# Send each order to the SQS queue
for order in orders:
    sqs.send_message(QueueUrl=q_url, MessageBody=json.dumps(order))
    print("Sent:", order["order_id"])

print("\n--- All orders sent successfully ---")
```

## Complete the Message Consumer

Excellent work mastering the producer side of messaging! Now it's time to complete the consumer pattern by receiving and processing messages from your SQS queue.

In this exercise, you'll work with code that has the basic consumer structure set up, but the actual message processing logic is incomplete. The code creates a queue, sends some sample orders to it, and has a processing loop ready, but several key pieces are missing.

Your task is to complete the consumer implementation by filling in three critical components:

* Convert the raw message body from JSON format back to a Python dictionary
* Call the provided `process_order()` function to process each message
* Delete processed messages from the queue to prevent duplicate processing

Look for the TODO comments in the code — they will guide you to the exact spots where you need to add the missing lines. The `process_order()` function is already provided and works correctly.

Once you complete this exercise, you'll have mastered the full SQS consumer pattern and will understand how proper message handling prevents processing the same message multiple times.

```python
import boto3, uuid, json, time, os

sqs = boto3.client('sqs')
sns = boto3.client('sns')

def create_queue_and_topic():
    q_name = f"orders-{uuid.uuid4().hex[:8]}"
    t_name = f"order-events-{uuid.uuid4().hex[:8]}"
    q_url = sqs.create_queue(QueueName=q_name)["QueueUrl"]
    topic_arn = sns.create_topic(Name=t_name)["TopicArn"]
    print("Created Queue URL:", q_url)
    print("Created Topic ARN:", topic_arn)
    return q_url, topic_arn

def send_orders(queue_url):
    orders = [{"order_id": "o-001", "amount": 42.0}, {"order_id": "o-002", "amount": 99.5}]
    for o in orders:
        sqs.send_message(QueueUrl=queue_url, MessageBody=json.dumps(o))
        print("Sent:", o["order_id"])

def process_order(body):
    return {"order_id": body["order_id"], "status": "processed", "amount": body["amount"]}

def consume_messages(queue_url, topic_arn):
    deadline = time.time() + 20
    while time.time() < deadline:
        resp = sqs.receive_message(QueueUrl=queue_url, MaxNumberOfMessages=5, WaitTimeSeconds=5)
        for m in resp.get("Messages", []):
            # TODO: Convert the message body from JSON string to Python dictionary
            # TODO: Call the process_order function with the converted message body
            # TODO: Delete the processed message from the queue using its ReceiptHandle
            print("Processed:", "order_id_here")

if __name__ == "__main__":
    # Step 1: Create AWS resources
    queue_url, topic_arn = create_queue_and_topic()
    
    # Step 2: Send orders to queue
    print("\n--- Sending Orders ---")
    send_orders(queue_url)
    
    # Step 3: Process messages and publish results
    print("\n--- Processing Orders ---")
    consume_messages(queue_url, topic_arn)
    
    print("\n--- Complete ---")
```

Here is the completed code with the message body decoded, processed, and deleted from the queue:

```python
import boto3, uuid, json, time, os

sqs = boto3.client('sqs')
sns = boto3.client('sns')

def create_queue_and_topic():
    q_name = f"orders-{uuid.uuid4().hex[:8]}"
    t_name = f"order-events-{uuid.uuid4().hex[:8]}"
    q_url = sqs.create_queue(QueueName=q_name)["QueueUrl"]
    topic_arn = sns.create_topic(Name=t_name)["TopicArn"]
    print("Created Queue URL:", q_url)
    print("Created Topic ARN:", topic_arn)
    return q_url, topic_arn

def send_orders(queue_url):
    orders = [{"order_id": "o-001", "amount": 42.0}, {"order_id": "o-002", "amount": 99.5}]
    for o in orders:
        sqs.send_message(QueueUrl=queue_url, MessageBody=json.dumps(o))
        print("Sent:", o["order_id"])

def process_order(body):
    return {"order_id": body["order_id"], "status": "processed", "amount": body["amount"]}

def consume_messages(queue_url, topic_arn):
    deadline = time.time() + 20
    while time.time() < deadline:
        resp = sqs.receive_message(QueueUrl=queue_url, MaxNumberOfMessages=5, WaitTimeSeconds=5)
        for m in resp.get("Messages", []):
            body = json.loads(m["Body"])
            result = process_order(body)
            sqs.delete_message(QueueUrl=queue_url, ReceiptHandle=m["ReceiptHandle"])
            print("Processed:", result["order_id"])

if __name__ == "__main__":
    # Step 1: Create AWS resources
    queue_url, topic_arn = create_queue_and_topic()
    
    # Step 2: Send orders to queue
    print("\n--- Sending Orders ---")
    send_orders(queue_url)
    
    # Step 3: Process messages and publish results
    print("\n--- Processing Orders ---")
    consume_messages(queue_url, topic_arn)
    
    print("\n--- Complete ---")
```

## Complete the Messaging Pipeline

Perfect! You've built a solid SQS consumer that handles message processing beautifully. Now it's time to connect the final piece of the messaging pipeline by adding SNS publishing functionality.

Your current consumer receives messages, processes them, and cleans them up properly, but the processed results aren't being shared with other parts of your system yet. This is where SNS becomes essential for creating truly event-driven applications.

Your task is to add the missing SNS publishing step that will notify downstream systems about completed orders. You need to add one line of code that publishes the processed result to the SNS topic using the correct parameters.

Look for the TODO comment in the `consume_messages()` function — it shows you exactly where to add the SNS publish call. This single addition will transform your consumer into a complete producer-consumer pipeline that demonstrates the full power of AWS messaging services.

```python
import boto3, uuid, json, time, os

sqs = boto3.client('sqs')
sns = boto3.client('sns')

def consume_messages(queue_url, topic_arn):
    deadline = time.time() + 20
    while time.time() < deadline:
        resp = sqs.receive_message(QueueUrl=queue_url, MaxNumberOfMessages=5, WaitTimeSeconds=5)
        for m in resp.get("Messages", []):
            body = json.loads(m["Body"])
            result = process_order(body)
            # TODO: Publish the processed result to the SNS topic
            sqs.delete_message(QueueUrl=queue_url, ReceiptHandle=m["ReceiptHandle"])
            print("Processed and published:", result["order_id"])

def create_queue_and_topic():
    q_name = f"orders-{uuid.uuid4().hex[:8]}"
    t_name = f"order-events-{uuid.uuid4().hex[:8]}"
    q_url = sqs.create_queue(QueueName=q_name)["QueueUrl"]
    topic_arn = sns.create_topic(Name=t_name)["TopicArn"]
    print("Created Queue URL:", q_url)
    print("Created Topic ARN:", topic_arn)
    return q_url, topic_arn

def send_orders(queue_url):
    orders = [{"order_id": "o-001", "amount": 42.0}, {"order_id": "o-002", "amount": 99.5}]
    for o in orders:
        sqs.send_message(QueueUrl=queue_url, MessageBody=json.dumps(o))
        print("Sent:", o["order_id"])

def process_order(body):
    return {"order_id": body["order_id"], "status": "processed", "amount": body["amount"]}

if __name__ == "__main__":
    # Step 1: Create AWS resources
    queue_url, topic_arn = create_queue_and_topic()
    
    # Step 2: Send orders to queue
    print("\n--- Sending Orders ---")
    send_orders(queue_url)
    
    # Step 3: Process messages and publish results
    print("\n--- Processing Orders ---")
    consume_messages(queue_url, topic_arn)
    
    print("\n--- Complete ---")
```

Here is the completed code with the SNS publish call added:

```python
import boto3, uuid, json, time, os

sqs = boto3.client('sqs')
sns = boto3.client('sns')

def consume_messages(queue_url, topic_arn):
    deadline = time.time() + 20
    while time.time() < deadline:
        resp = sqs.receive_message(QueueUrl=queue_url, MaxNumberOfMessages=5, WaitTimeSeconds=5)
        for m in resp.get("Messages", []):
            body = json.loads(m["Body"])
            result = process_order(body)
            sns.publish(TopicArn=topic_arn, Message=json.dumps(result))
            sqs.delete_message(QueueUrl=queue_url, ReceiptHandle=m["ReceiptHandle"])
            print("Processed and published:", result["order_id"])

def create_queue_and_topic():
    q_name = f"orders-{uuid.uuid4().hex[:8]}"
    t_name = f"order-events-{uuid.uuid4().hex[:8]}"
    q_url = sqs.create_queue(QueueName=q_name)["QueueUrl"]
    topic_arn = sns.create_topic(Name=t_name)["TopicArn"]
    print("Created Queue URL:", q_url)
    print("Created Topic ARN:", topic_arn)
    return q_url, topic_arn

def send_orders(queue_url):
    orders = [{"order_id": "o-001", "amount": 42.0}, {"order_id": "o-002", "amount": 99.5}]
    for o in orders:
        sqs.send_message(QueueUrl=queue_url, MessageBody=json.dumps(o))
        print("Sent:", o["order_id"])

def process_order(body):
    return {"order_id": body["order_id"], "status": "processed", "amount": body["amount"]}

if __name__ == "__main__":
    # Step 1: Create AWS resources
    queue_url, topic_arn = create_queue_and_topic()
    
    # Step 2: Send orders to queue
    print("\n--- Sending Orders ---")
    send_orders(queue_url)
    
    # Step 3: Process messages and publish results
    print("\n--- Processing Orders ---")
    consume_messages(queue_url, topic_arn)
    
    print("\n--- Complete ---")
```

## Fix the Order Processing Logic

Fantastic work building your complete messaging pipeline! Now you're ready to tackle more sophisticated business logic within your message processing system.

In this exercise, you'll work with an order processing system that needs to categorize orders based on their value. The system should assign express status to high-value orders (above $50) and standard status to regular orders ($50 or below). This type of conditional processing is common in real-world applications, where different message types require different handling.

The infrastructure (SQS, SNS, message handling) all works perfectly — the problem is in the business logic. Look carefully at the `process_order()` function and fix the conditional statement so that orders with amounts above $50 get express status, while orders of $50 or below get standard status.

The test data include orders with different amounts, so you'll be able to see if your fix works correctly when the processed results are published to SNS. Once you solve this, you'll understand how to implement complex business rules in event-driven messaging systems.

```python
import boto3, uuid, json, time, os

sqs = boto3.client('sqs')
sns = boto3.client('sns')

def process_order(body):
    if body["amount"] < 50.0:
        status = "express"
    else:
        status = "standard"
    return {"order_id": body["order_id"], "status": status, "amount": body["amount"]}

def create_queue_and_topic():
    q_name = f"orders-{uuid.uuid4().hex[:8]}"
    t_name = f"order-events-{uuid.uuid4().hex[:8]}"
    q_url = sqs.create_queue(QueueName=q_name)["QueueUrl"]
    topic_arn = sns.create_topic(Name=t_name)["TopicArn"]
    print("Created Queue URL:", q_url)
    print("Created Topic ARN:", topic_arn)
    return q_url, topic_arn

def send_orders(queue_url):
    orders = [
        {"order_id": "o-001", "amount": 25.0},
        {"order_id": "o-002", "amount": 75.0},
        {"order_id": "o-003", "amount": 50.0},
        {"order_id": "o-004", "amount": 120.0}
    ]
    for o in orders:
        sqs.send_message(QueueUrl=queue_url, MessageBody=json.dumps(o))
        print("Sent:", o["order_id"], "- Amount:", o["amount"])

def consume_messages(queue_url, topic_arn):
    deadline = time.time() + 20
    while time.time() < deadline:
        resp = sqs.receive_message(QueueUrl=queue_url, MaxNumberOfMessages=5, WaitTimeSeconds=5)
        for m in resp.get("Messages", []):
            body = json.loads(m["Body"])
            result = process_order(body)
            sns.publish(TopicArn=topic_arn, Message=json.dumps(result))
            sqs.delete_message(QueueUrl=queue_url, ReceiptHandle=m["ReceiptHandle"])
            print("Processed:", result["order_id"], "- Status:", result["status"])

if __name__ == "__main__":
    # Step 1: Create AWS resources
    queue_url, topic_arn = create_queue_and_topic()
    
    # Step 2: Send orders to queue
    print("\n--- Sending Orders ---")
    send_orders(queue_url)
    
    # Step 3: Process messages and publish results
    print("\n--- Processing Orders ---")
    consume_messages(queue_url, topic_arn)
    
    print("\n--- Complete ---")
```

Here is the completed code with the conditional in `process_order()` fixed so amounts above $50 are `"express"` and amounts of $50 or below are `"standard"`:

```python
import boto3, uuid, json, time, os

sqs = boto3.client('sqs')
sns = boto3.client('sns')

def process_order(body):
    if body["amount"] > 50.0:
        status = "express"
    else:
        status = "standard"
    return {"order_id": body["order_id"], "status": status, "amount": body["amount"]}

def create_queue_and_topic():
    q_name = f"orders-{uuid.uuid4().hex[:8]}"
    t_name = f"order-events-{uuid.uuid4().hex[:8]}"
    q_url = sqs.create_queue(QueueName=q_name)["QueueUrl"]
    topic_arn = sns.create_topic(Name=t_name)["TopicArn"]
    print("Created Queue URL:", q_url)
    print("Created Topic ARN:", topic_arn)
    return q_url, topic_arn

def send_orders(queue_url):
    orders = [
        {"order_id": "o-001", "amount": 25.0},
        {"order_id": "o-002", "amount": 75.0},
        {"order_id": "o-003", "amount": 50.0},
        {"order_id": "o-004", "amount": 120.0}
    ]
    for o in orders:
        sqs.send_message(QueueUrl=queue_url, MessageBody=json.dumps(o))
        print("Sent:", o["order_id"], "- Amount:", o["amount"])

def consume_messages(queue_url, topic_arn):
    deadline = time.time() + 20
    while time.time() < deadline:
        resp = sqs.receive_message(QueueUrl=queue_url, MaxNumberOfMessages=5, WaitTimeSeconds=5)
        for m in resp.get("Messages", []):
            body = json.loads(m["Body"])
            result = process_order(body)
            sns.publish(TopicArn=topic_arn, Message=json.dumps(result))
            sqs.delete_message(QueueUrl=queue_url, ReceiptHandle=m["ReceiptHandle"])
            print("Processed:", result["order_id"], "- Status:", result["status"])

if __name__ == "__main__":
    # Step 1: Create AWS resources
    queue_url, topic_arn = create_queue_and_topic()
    
    # Step 2: Send orders to queue
    print("\n--- Sending Orders ---")
    send_orders(queue_url)
    
    # Step 3: Process messages and publish results
    print("\n--- Processing Orders ---")
    consume_messages(queue_url, topic_arn)
    
    print("\n--- Complete ---")
```